In [1]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

In [5]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset
# ========================
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (Random Forest)
# ========================
mlp_global = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)

mlp_global.fit(X_train, y_train)

def predict_fn(X):
    return mlp_global.predict(X)
# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

# ========================
# Fonction pour mesurer la fidélité avec train/test de perturbations
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=5000, num_samples_test=2000, metric="r2"):
    """
    Calcule la fidélité d'un modèle local par rapport au modèle global
    sur des perturbations différentes pour l'entraînement et le test.
    """
    num_features = instance.shape[0]

    # --- Perturbations pour entraîner le modèle local ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Perturbations pour tester la fidélité (nouvelles, non vues) ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    # Calcul de la fidélité
    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        return mean_squared_error(y_global_test, y_local_test)
    else:
        raise ValueError("metric doit être 'r2' ou 'mse'")

# ========================
# Comparer les 3 modèles sur plusieurs instances du test
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], gammas=[100,1]),
                             predict_fn)
    )

# Moyenne sur toutes les instances
print("=== Fidélité moyenne des modèles locaux (MSE) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


mean_leaf_values shape: (29, 1, 1)
self.L: 29 self.C: 1
mean_leaf_values shape after squeeze: (29, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  
=== Fidélité moyenne des modèles locaux (MSE) ===
LinearRegression: 0.376
DecisionTree: 0.966
NDT: 0.910


## Diabetes Dataset

In [4]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset
# ========================
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (Random Forest)
# ========================
mlp_global = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)

mlp_global.fit(X_train, y_train)

def predict_fn(X):
    return mlp_global.predict(X)
# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

# ========================
# Fonction pour mesurer la fidélité avec train/test de perturbations
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=5000, num_samples_test=2000, metric="r2"):
    """
    Calcule la fidélité d'un modèle local par rapport au modèle global
    sur des perturbations différentes pour l'entraînement et le test.
    """
    num_features = instance.shape[0]

    # --- Perturbations pour entraîner le modèle local ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Perturbations pour tester la fidélité (nouvelles, non vues) ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    # Calcul de la fidélité
    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        return mean_squared_error(y_global_test, y_local_test)
    else:
        raise ValueError("metric doit être 'r2' ou 'mse'")

# ========================
# Comparer les 3 modèles sur plusieurs instances du test
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], gammas=[100,1]),
                             predict_fn)
    )

# Moyenne sur toutes les instances
print("=== Fidélité moyenne des modèles locaux (R2) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
=== Fidélité moyenne des modèles locaux (R2) ===
LinearRegression: 0.884
DecisionTree: 0.531
NDT: 0.785


## Iris Dataset

In [6]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import r2_score

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Dataset Iris
# ========================
data = load_breast_cancer()
X = data.data
y = data.target.astype(float)
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

# ========================
# 2. Modèle global
# ========================
mlp_global = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    max_iter=2000,
    random_state=42
)
mlp_global.fit(X_train, y_train)

def predict_fn(X):
    return mlp_global.predict(X).astype(float)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="classification"
)

explainer_ndt = LimeNdtExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="classification"
)

# ========================
# 4. Fidélité locale (UNE instance)
# ========================
def fidelity_local_model(
    instance,
    explainer,
    local_model,
    predict_fn,
    num_samples_train=5000,
    num_samples_test=2000,
    metric="r2"
):
    num_features = instance.shape[0]

    # --- Train perturbations ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)

    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Test perturbations ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)

    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        from sklearn.metrics import mean_squared_error
        return mean_squared_error(y_global_test, y_local_test)

# ========================
# 5. Fidélité sur 100 instances
# ========================
n_instances = 100
rng = np.random.default_rng(42)
indices = rng.choice(len(X_test), size=n_instances, replace=False)

results = {
    "LinearRegression": [],
    "DecisionTree": [],
    "NDT": []
}

for idx in indices:
    instance = X_test[idx]

    results["LinearRegression"].append(
        fidelity_local_model(
            instance,
            explainer_classic,
            LinearRegression(),
            predict_fn
        )
    )

    results["DecisionTree"].append(
        fidelity_local_model(
            instance,
            explainer_ndt,
            DecisionTreeWrapper(),
            predict_fn
        )
    )

    results["NDT"].append(
        fidelity_local_model(
            instance,
            explainer_ndt,
            NDTRegressorWrapper(D=X_train.shape[1], gammas=[100, 1]),
            predict_fn
        )
    )

# ========================
# 6. Résultats
# ========================
print("\n=== Fidélité des modèles locaux (R²) — 100 instances ===")
for model_name, scores in results.items():
    scores = np.array(scores)
    print(f"{model_name}: {scores.mean():.3f} ± {scores.std():.3f}")


mean_leaf_values shape: (29, 1, 1)
self.L: 29 self.C: 1
mean_leaf_values shape after squeeze: (29, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
mean_leaf_values shape: (30, 1, 1)
self.L: 30 self.C: 1
mean_leaf_values shape after squeeze: (30, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
mean_leaf_values shape: (30, 1, 1)
self.L: 30 self.C: 1
mean_leaf_values shape after squeeze: (30, 1)


...

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
mean_leaf_values shape: (31, 1, 1)
self.L: 31 self.C: 1
mean_leaf_values shape after squeeze: (31, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
mean_leaf_values shape: (31, 1, 1)
self.L: 31 self.C: 1
mean_leaf_values shape after squeeze: (31, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

=== Fidélité des modèles locaux (R²) — 100 instances ===
LinearRegression: 0.527 ± 0.085
DecisionTree: 0.686 ± 0.049
NDT: 0.785 ± 0.031


## Wine Dataset

In [19]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import r2_score

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset Iris
# ========================
data = load_wine()
X = data.data
y = data.target.astype(float)  # convert to float pour la régression
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLPClassifier)
# ========================
mlp_global = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)
mlp_global.fit(X_train, y_train)

def predict_fn(X):
    # retourner les labels comme float pour la fidélité locale
    return mlp_global.predict(X).astype(float)

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification'  # pour LinearRegression local
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification'  # on utilise regressor wrapper
)

# ========================
# Fonction fidélité locale
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=5000, num_samples_test=2000, metric="r2"):
    num_features = instance.shape[0]

    # --- Train perturbations ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Test perturbations ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        from sklearn.metrics import mean_squared_error
        return mean_squared_error(y_global_test, y_local_test)

# ========================
# Comparer les modèles locaux
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
                             predict_fn)
    )

print("=== Fidélité moyenne des modèles locaux (R²) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


mean_leaf_values shape: (30, 1, 1)
self.L: 30 self.C: 1
mean_leaf_values shape after squeeze: (30, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
=== Fidélité moyenne des modèles locaux (R²) ===
LinearRegression: 0.369
DecisionTree: 0.250
NDT: 0.364


## Breast Cancer Dataset

In [20]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import r2_score

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset Iris
# ========================
data = load_breast_cancer()
X = data.data
y = data.target.astype(float)  # convert to float pour la régression
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLPClassifier)
# ========================
mlp_global = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)
mlp_global.fit(X_train, y_train)

def predict_fn(X):
    # retourner les labels comme float pour la fidélité locale
    return mlp_global.predict(X).astype(float)

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification' 
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification'
)

# ========================
# Fonction fidélité locale
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=5000, num_samples_test=2000, metric="r2"):
    num_features = instance.shape[0]

    # --- Train perturbations ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Test perturbations ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        from sklearn.metrics import mean_squared_error
        return mean_squared_error(y_global_test, y_local_test)

# ========================
# Comparer les modèles locaux
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
                             predict_fn)
    )

print("=== Fidélité moyenne des modèles locaux (R²) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
=== Fidélité moyenne des modèles locaux (R²) ===
LinearRegression: 0.566
DecisionTree: 0.702
NDT: 0.888


## Digits Dataset

In [3]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import r2_score

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset Iris
# ========================
data = load_digits()
X = data.data
y = data.target.astype(float)  # convert to float pour la régression
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLPClassifier)
# ========================
mlp_global = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32, 32),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)
mlp_global.fit(X_train, y_train)

def predict_fn(X):
    # retourner les labels comme float pour la fidélité locale
    return mlp_global.predict(X).astype(float)

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification'  # pour LinearRegression local
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification'  # on utilise regressor wrapper
)

# ========================
# Fonction fidélité locale
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=50000, num_samples_test=500, metric="r2"):
    num_features = instance.shape[0]

    # --- Train perturbations ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Test perturbations ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        from sklearn.metrics import mean_squared_error
        return mean_squared_error(y_global_test, y_local_test)

# ========================
# Comparer les modèles locaux
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
                             predict_fn)
    )

print("=== Fidélité moyenne des modèles locaux (R²) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
=== Fidélité moyenne des modèles locaux (R²) ===
LinearRegression: 0.113
DecisionTree: -0.895
NDT: 0.586


## Covtype Dataset

In [3]:
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import r2_score

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset Iris
# ========================
data = fetch_covtype()
X = data.data[:2000]
y = data.target[:2000].astype(float)  # convert to float pour la régression
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLPClassifier)
# ========================
mlp_global = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)
mlp_global.fit(X_train, y_train)

def predict_fn(X):
    # retourner les labels comme float pour la fidélité locale
    return mlp_global.predict(X).astype(float)

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification'  # pour LinearRegression local
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='classification'  # on utilise regressor wrapper
)

# ========================
# Fonction fidélité locale
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=5000, num_samples_test=500, metric="r2"):
    num_features = instance.shape[0]

    # --- Train perturbations ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Test perturbations ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        from sklearn.metrics import mean_squared_error
        return mean_squared_error(y_global_test, y_local_test)

# ========================
# Comparer les modèles locaux
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
                             predict_fn)
    )

print("=== Fidélité moyenne des modèles locaux (R²) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


mean_leaf_values shape: (31, 1, 1)
self.L: 31 self.C: 1
mean_leaf_values shape after squeeze: (31, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
=== Fidélité moyenne des modèles locaux (R²) ===
LinearRegression: 0.206
DecisionTree: 0.177
NDT: 0.476


## Ames Housing Dataset

In [7]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset
# ========================
data = fetch_openml(name='house_prices', as_frame=True)
X = data.data.select_dtypes(include=[np.number]).dropna(axis=1)
y = data.target.astype(float)
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (Random Forest)
# ========================
mlp_global = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)

mlp_global.fit(X_train, y_train)

def predict_fn(X):
    return mlp_global.predict(X)
# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

# ========================
# Fonction pour mesurer la fidélité avec train/test de perturbations
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=5000, num_samples_test=2000, metric="r2"):
    """
    Calcule la fidélité d'un modèle local par rapport au modèle global
    sur des perturbations différentes pour l'entraînement et le test.
    """
    num_features = instance.shape[0]

    # --- Perturbations pour entraîner le modèle local ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Perturbations pour tester la fidélité (nouvelles, non vues) ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    # Calcul de la fidélité
    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        return mean_squared_error(y_global_test, y_local_test)
    else:
        raise ValueError("metric doit être 'r2' ou 'mse'")

# ========================
# Comparer les 3 modèles sur plusieurs instances du test
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test.iloc[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], gammas=[100,1]),
                             predict_fn)
    )

# Moyenne sur toutes les instances
print("=== Fidélité moyenne des modèles locaux (R2) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


C:\Users\DELL\AppData\Local\Temp\ipykernel_8964\358985802.py:81: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean = instance[i]
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MLPRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Temp\ipykernel_8964\358985802.py:90: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean = instance[i]
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MLPRegressor was f

mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
=== Fidélité moyenne des modèles locaux (R2) ===
LinearRegression: 0.860
DecisionTree: 0.446
NDT: 0.667


C:\Users\DELL\AppData\Local\Temp\ipykernel_8964\358985802.py:90: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean = instance[i]
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MLPRegressor was fitted with feature names
  warnings.warn(
